# 🧠 NS-ARC Small Slot-JEPA Experiment

Based on Harmonic Frequency Initialization, Patch/Decoder MLPs, and deep object compositionality theories.

Principle: **Factorize → Stabilize → Disentangle → Reason**

## 1. Setup & Dependencies

In [ ]:
import sys, os, subprocess
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def run(cmd):
    print(f"Executing: {cmd}")
    subprocess.run(cmd, shell=True, check=True)

# Try Kaggle Path
KAGGLE_REPO_PATH = '/kaggle/working/Model-Jepa'
if os.path.exists(KAGGLE_REPO_PATH):
    print('✅ Kaggle detected.')
    if KAGGLE_REPO_PATH not in sys.path:
        sys.path.insert(0, KAGGLE_REPO_PATH)
else:
    sys.path.insert(0, os.path.abspath('.'))
    print('✅ Local mode.')

if torch.cuda.is_available(): DEVICE = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available(): DEVICE = 'mps'
else: DEVICE = 'cpu'
print(f'Device: {DEVICE}')


## 2. Model Configurations (Small Architecture)

In [ ]:
BASE_CFG = {
    'device': DEVICE,
    'in_channels': 1,
    'input_dim': 64,
    'patch_size': 2,
    'hidden_dim': 128,          # Scaled down to 128
    'latent_dim': 128,
    'num_slots': 10,            # K=10 Maximum objects
    'slot_iters': 5,            # Iterations bumped to 5 for complex topologies
    'slot_temperature': 0.1,    
    'vocab_size': 10,           # ARC Colors
    'grid_size': 30,            # 30x30 output
    'focal_gamma': 2.0,         
}


## 3. Harmonic Slot Attention & Core Backbone

In [ ]:
class HarmonicSlotEncoder(nn.Module):
    """
    Object-Centric Encoder using Iterative Slot Attention.
    Initialized with Deterministic Fourier Frequencies to enforce structural uniqueness per slot.
    """
    def __init__(self, config):
        super().__init__()
        self.device = config['device']
        self.embed_dim = config['hidden_dim']
        self.num_slots = config['num_slots']
        self.slot_iters = config['slot_iters']
        self.temperature = config['slot_temperature']
        
        # 1. Feature Backbone (CNN)
        self.patch_embed = nn.Conv2d(config['in_channels'], self.embed_dim, kernel_size=config['patch_size'], stride=config['patch_size'])
        self.patch_pos_embed = nn.Parameter(torch.randn(1, 1024, self.embed_dim)) # Over-provisioned
        
        # 2. Patch MLP (Addresses Deep Non-Linearity Review)
        self.patch_mlp = nn.Sequential(
            nn.Linear(self.embed_dim, self.embed_dim * 2),
            nn.LayerNorm(self.embed_dim * 2),
            nn.GELU(),
            nn.Linear(self.embed_dim * 2, self.embed_dim)
        )
        
        # 3. Harmonic Object Priors (Addresses Deterministic Frequency Prior Idea)
        # Instead of random noise N(mu, sigma), we hard-code sin/cosine frequencies across the 10 slots
        # to hunt for distinct topologies implicitly.
        self.harmonic_priors = nn.Parameter(self._build_harmonic_priors(self.num_slots, self.embed_dim), requires_grad=False)
        self.slot_mu = nn.Parameter(torch.zeros(1, 1, self.embed_dim)) # Learnable offset
        self.slot_logsigma = nn.Parameter(torch.ones(1, 1, self.embed_dim) * -2.0)
        
        # 4. Attention Projections
        self.norm_inputs = nn.LayerNorm(self.embed_dim)
        self.norm_slots  = nn.LayerNorm(self.embed_dim)
        self.norm_mlp    = nn.LayerNorm(self.embed_dim)
        
        self.q_proj = nn.Linear(self.embed_dim, self.embed_dim, bias=False)
        self.k_proj = nn.Linear(self.embed_dim, self.embed_dim, bias=False)
        self.v_proj = nn.Linear(self.embed_dim, self.embed_dim, bias=False)
        self.gru = nn.GRUCell(self.embed_dim, self.embed_dim)
        
        self.slot_mlp = nn.Sequential(
            nn.Linear(self.embed_dim, self.embed_dim * 2),
            nn.GELU(),
            nn.Linear(self.embed_dim * 2, self.embed_dim)
        )
        
        self.to(self.device)

    def _build_harmonic_priors(self, num_slots, dim):
        priors = torch.zeros(1, num_slots, dim)
        position = torch.arange(0, num_slots, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, dim, 2).float() * (-np.log(10000.0) / dim))
        priors[0, :, 0::2] = torch.sin(position * div_term)
        priors[0, :, 1::2] = torch.cos(position * div_term)
        return priors

    def forward(self, inputs):
        img = inputs["state"].float().to(self.device)
        B = img.shape[0]
        
        # CNN -> Patch Map
        x = self.patch_embed(img) # [B, D, H', W']
        _, _, H, W = x.shape
        x = x.flatten(2).transpose(1, 2) # [B, N, D]
        N = x.shape[1]
        
        # Add Soft Position and pass through deep NON-LINEAR MLP
        x = x + self.patch_pos_embed[:, :N, :]
        x = self.patch_mlp(x) # [B, N, D]
        x = self.norm_inputs(x)
        
        k = self.k_proj(x)
        v = self.v_proj(x)
        
        # Initialize Slots: Harmonic Fourier Features + Minor stochastic exploration noise
        slots = self.harmonic_priors.expand(B, -1, -1) + self.slot_mu
        if self.training:
            slots = slots + torch.randn_like(slots) * self.slot_logsigma.exp()
            
        # Iterative Routing via Softmax Competition
        for _ in range(self.slot_iters):
            slots_prev = slots
            slots_norm = self.norm_slots(slots)
            q = self.q_proj(slots_norm)

            attn_logits = torch.bmm(k, q.transpose(1, 2)) * (self.embed_dim ** -0.5)
            attn = F.softmax(attn_logits / self.temperature, dim=-1) # Compete over slots!
            attn_norm = attn + 1e-8
            attn_norm = attn_norm / attn_norm.sum(dim=1, keepdim=True)

            updates = torch.bmm(attn_norm.transpose(1, 2), v)
            slots = self.gru(updates.reshape(-1, self.embed_dim), slots_prev.reshape(-1, self.embed_dim)).reshape(B, self.num_slots, self.embed_dim)
            slots = slots + self.slot_mlp(self.norm_mlp(slots))

        masks = attn.transpose(1, 2).view(B, self.num_slots, H, W).detach()
        return {"latent": slots, "masks": masks, "slot_logsigma": self.slot_logsigma}


## 4. Decoder & Feedback Masking

In [ ]:
class SoftPositionEmbed(nn.Module):
    """Explicit position signal so slots differentiate colors from coords."""
    def __init__(self, hidden_size, resolution):
        super().__init__()
        self.dense = nn.Linear(4, hidden_size)
        self.register_buffer("grid", self._build_grid(resolution))

    @staticmethod
    def _build_grid(resolution):
        ranges = [torch.linspace(0.0, 1.0, steps=r) for r in resolution]
        grid = torch.meshgrid(*ranges, indexing="ij")
        grid = torch.stack(grid, dim=-1)
        grid = torch.cat([grid, 1.0 - grid], dim=-1)
        return grid.unsqueeze(0)

    def forward(self, x):
        pos = self.dense(self.grid).permute(0, 3, 1, 2)
        return x + pos

class HarmonicDecoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.device = config['device']
        self.embed_dim = config['hidden_dim']
        self.vocab_size = config['vocab_size']
        self.grid_size = config['grid_size']
        
        self.decoder_pos = SoftPositionEmbed(self.embed_dim, (self.grid_size, self.grid_size))
        
        self.spatial_broadcast = nn.Sequential(
            nn.ConvTranspose2d(self.embed_dim, 64, kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=5, stride=1, padding=2),
            nn.ReLU()
        )
        
        # Deep MLPs for Alpha and Color Mask logic
        self.color_mlp = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1),
            nn.SiLU(),
            nn.Conv2d(64, self.vocab_size, 3, padding=1)
        )
        
        self.alpha_mlp = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1),
            nn.SiLU(),
            nn.Conv2d(64, 1, 3, padding=1)
        )
        self.to(self.device)

    def forward(self, inputs):
        z = inputs["latent"] # [B, S, D]
        B, num_slots, D = z.shape
        H, W = self.grid_size, self.grid_size
        
        z_tiled = z.view(B * num_slots, D, 1, 1).expand(-1, -1, H, W)
        z_tiled = self.decoder_pos(z_tiled)
        
        decoded_features = self.spatial_broadcast(z_tiled)
        
        colors = self.color_mlp(decoded_features).view(B, num_slots, self.vocab_size, H, W)
        alphas = self.alpha_mlp(decoded_features).view(B, num_slots, 1, H, W)
        
        alphas_normalized = F.softmax(alphas, dim=1) # Composite competition
        reconstruction = torch.sum(alphas_normalized * colors, dim=1)
        
        return {"reconstruction": reconstruction, "alphas": alphas_normalized}

    def loss(self, inputs, outputs):
        target = inputs["state"].to(self.device).long().squeeze()
        if target.dim() == 2: target = target.unsqueeze(0)
        recon_logits = outputs["reconstruction"] 
        
        # 1. Focal Reconstruction Feedback
        ce_loss = F.cross_entropy(recon_logits, target, reduction='none')
        pt = torch.exp(-ce_loss)
        recon_loss = (((1 - pt) ** 2.0) * ce_loss).mean()
        
        # 2. Orthogonality Diversity Feedback (Push slots apart)
        z = inputs["latent"] 
        z_norm = F.normalize(z, dim=-1)
        sim = torch.bmm(z_norm, z_norm.transpose(1, 2))
        S = sim.size(1)
        eye = torch.eye(S, device=self.device).unsqueeze(0).bool()
        sim_off = sim.masked_fill(eye, 0.0)
        ortho_loss = F.relu(sim_off.abs() - 0.3).mean() # Strict Hinge > 0.3
        
        total_loss = recon_loss + (0.5 * ortho_loss)
        return {"loss": total_loss, "recon_loss": recon_loss, "ortho_loss": ortho_loss}


## 5. Formalized Logic Definition complete! You may now implement training loop below.